In [1]:
import numpy as np
import pandas as pd
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import json
from tensorflow.keras.models import load_model


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
def mape(y_true, y_pred):
    y_true = np.array(y_true).reshape(-1)
    y_pred = np.array(y_pred).reshape(-1)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


In [3]:
def create_sequences(X, lookback=24):
    Xs = []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
    return np.array(Xs)


In [4]:
DATA_PATH = "../data/processed/featured_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

with open("../saved_models/features.json") as f:
    feature_names = json.load(f)

TARGET = "load_actual"

split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:]

X_test_raw = test_df[feature_names]
y_test = test_df[TARGET].values

lookback = 24


In [5]:
def prepare_dl_test(model_name):
    scaler_X = joblib.load(f"../saved_models/{model_name}_scaler_X.pkl")
    scaler_y = joblib.load(f"../saved_models/{model_name}_scaler_y.pkl")

    X_scaled = scaler_X.transform(X_test_raw)
    y_scaled = scaler_y.transform(y_test.reshape(-1,1))

    X_seq = create_sequences(X_scaled, lookback)
    y_seq = y_scaled[lookback:]

    return X_seq, y_seq, scaler_y


In [6]:
models = {
    "RandomForest": {"type": "pickle", "path": "../saved_models/rf_model.pkl"},
    "SVM": {"type": "svm", "path": "../saved_models/svm_model.pkl"},
    "XGBoost": {"type": "pickle", "path": "../saved_models/xgb_model.pkl"},
    "CatBoost": {"type": "cat", "path": "../saved_models/catboost_model.cbm"},
    "ARIMA": {"type": "arima", "path": "../saved_models/arima_model.pkl"},

    # Deep Learning
    "LSTM": {"type": "keras_dl", "path": "../saved_models/lstm_model.h5"},
    "GRU": {"type": "keras_dl", "path": "../saved_models/gru_model.h5"},
    "CNN": {"type": "keras_dl", "path": "../saved_models/cnn_model.h5"},
    "CNN_LSTM": {"type": "keras_dl", "path": "../saved_models/cnn_lstm_model.h5"},
    "BPNN": {"type": "bpnn", "path": "../saved_models/bpnn_model.h5"}
}


In [7]:
results = []

for name, info in models.items():
    try:
        print(f"\nEvaluating {name}...")

        if info["type"] == "pickle":
            model = joblib.load(info["path"])
            preds = model.predict(X_test_raw)
            y_use = y_test

        elif info["type"] == "svm":
            model = joblib.load(info["path"])
            scaler_X = joblib.load("../saved_models/svm_scaler.pkl")
            X_scaled = scaler_X.transform(X_test_raw)
            preds = model.predict(X_scaled)
            y_use = y_test

        elif info["type"] == "cat":
            from catboost import CatBoostRegressor
            model = CatBoostRegressor()
            model.load_model(info["path"])
            preds = model.predict(X_test_raw)
            y_use = y_test

        elif info["type"] == "arima":
            model = joblib.load(info["path"])
            preds = model.forecast(steps=len(y_test))
            y_use = y_test

        elif info["type"] == "keras_dl":
            X_seq, y_seq, scaler_y = prepare_dl_test(name.lower())
            model = load_model(info["path"], compile=False)
            preds = scaler_y.inverse_transform(model.predict(X_seq))
            y_use = y_test[lookback:]

        elif info["type"] == "bpnn":
            scaler_X = joblib.load("../saved_models/bpnn_scaler_X.pkl")
            scaler_y = joblib.load("../saved_models/bpnn_scaler_y.pkl")
            X_scaled = scaler_X.transform(X_test_raw)
            model = load_model(info["path"], compile=False)
            preds = scaler_y.inverse_transform(model.predict(X_scaled))
            y_use = y_test

        mae = mean_absolute_error(y_use, preds)
        rmse = sqrt(mean_squared_error(y_use, preds))
        mape_val = mape(y_use, preds)

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape_val
        })

    except Exception as e:
        print(f"❌ {name} failed: {e}")



Evaluating RandomForest...

Evaluating SVM...

Evaluating XGBoost...

Evaluating CatBoost...

Evaluating ARIMA...

Evaluating LSTM...


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Evaluating GRU...


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Evaluating CNN...
 64/273 ━━━━━━━━━━━━━━━━━━━━ 0s 802us/step

...

273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step

Evaluating CNN_LSTM...


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Evaluating BPNN...


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


273/273 ━━━━━━━━━━━━━━━━━━━━ 0s 542us/step


In [8]:
results_df = pd.DataFrame(results).sort_values("MAPE").reset_index(drop=True)
print("\n=== FINAL MODEL PERFORMANCE COMPARISON ===")
results_df



=== FINAL MODEL PERFORMANCE COMPARISON ===


,Model,MAE,RMSE,MAPE
0,XGBoost,199.632575,283.615741,0.709697
1,CatBoost,201.642373,278.160467,0.716598
2,RandomForest,226.270752,323.130654,0.799917
3,SVM,333.347251,474.809632,1.178418
4,GRU,350.948390,453.646248,1.273258
5,CNN_LSTM,453.221246,622.717327,1.609421
6,CNN,478.702358,659.471580,1.709070
7,LSTM,712.612815,923.162186,2.377231
8,BPNN,832.286343,988.364872,2.952234
9,ARIMA,5284.485489,6500.768778,17.052999


In [9]:
results_df.to_csv("../saved_models/model_comparison.csv", index=False)
